Chatbot And RAG Evaluation

Retrieval Augmented Generation (RAG) is a technique that enhances Large 
Language Models (LLMs) by providing them with relevant external knowledge. It has become one of the most widely used approaches for building LLM applications.

This tutorial will show you how to evaluate your RAG applications using LangSmith. You'll learn:

How to create test datasets

How to run your RAG application on those datasets

How to measure your application's performance using different evaluation metrics
Overview

A typical RAG evaluation workflow consists of three main steps:

Creating a dataset with questions and their expected answers

Running your RAG application on those questions

Using evaluators to measure how well your application performed, looking at factors like:
Answer relevance

Answer accuracy

Retrieval quality

For this tutorial, we'll create and evaluate a bot that answers questions about a few of Lilian Weng's insightful blog posts.

Chatbot Evaluation

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["LANGSMITH_TRACING"]="true"

In [5]:
LANGCHAIN_ENDPOINT="https://api.smith.langchain.com"

In [6]:
from langsmith import Client

client = Client()

# Define dataset: these are your test cases
dataset_name = "Chatbot EEvaluation"
dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)

{'example_ids': ['c2667162-9435-4b85-856f-8aee7819ed1f',
  'bae4e758-f185-48bf-bbe7-8aeae031eb5b',
  '58289eb4-8c0a-4673-b164-1fd8a88cef15',
  '58588a27-8586-4f76-a793-2745f1e9d4ed',
  'b6f95818-6a3a-41f8-bb80-d505d202c3fb'],
 'count': 5,
 'as_of': '2026-06-12T07:18:30.646497036Z'}

Define Metrics (LLM As A Judge)

In [7]:
from groq import Groq

# Initialize Groq client
groq_client = Groq()

eval_instructions = (
    "You are an expert professor specialized in grading students' answers to questions."
)

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content = f"""You are grading the following question:
{inputs['question']}

Here is the real answer:
{reference_outputs['answer']}

You are grading the following predicted answer:
{outputs['response']}

Respond with only CORRECT or INCORRECT.

Grade:
"""

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",  # Choose any Groq model
        temperature=0,
        messages=[
            {"role": "system", "content": eval_instructions},
            {"role": "user", "content": user_content}
        ]
    )

    grade = response.choices[0].message.content.strip().upper()

    return grade == "CORRECT"

In [8]:
## Concisions- checks whether the actual output is less than 2x the length of the expected result.

def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

Run Evaluations

In [9]:
default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."
def my_app(question: str, model: str = "llama-3.3-70b-versatile", instructions: str = default_instructions) -> str:
    return groq_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    ).choices[0].message.content

In [10]:
#call my_app for every datapoints

def ls_target(input:str)->dict:
    return {"response":my_app(input["question"])}

In [11]:
experiment_results=client.evaluate(
    ls_target,
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="llama-3.3-70b-versatile"
)

c:\Users\Omkar Kadam\Desktop\Agentic AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'llama-3.3-70b-versatile-0f16956b' at:
https://smith.langchain.com/o/87d68caf-a391-48ca-a2c6-1fc4f60b9445/datasets/bd82a737-7cd3-4400-8946-da906ee2513b/compare?selectedSessions=cff4bf66-9d40-422c-aa8e-05eb0dbff713




5it [00:03,  1.58it/s]


Evaluation For RAG

In [12]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# URLs
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

# Load documents
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

# Split documents
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250,
    chunk_overlap=0
)

doc_splits = text_splitter.split_documents(docs_list)

# Embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create vector store
vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=embedding_model
)

# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

C:\Users\Omkar Kadam\AppData\Local\Temp\ipykernel_21740\4054639926.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2817.67it/s]


In [13]:

retriever.invoke("what is agents")

[Document(id='007db4c1-3dea-4d35-88fb-159e21642a1c', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, 

In [14]:
from langchain.chat_models import init_chat_model
llm=init_chat_model("groq:llama-3.3-70b-versatile")
llm


ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000027F5923D310>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000027F5A71C050>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [15]:
from langsmith import traceable

#add decorator
@traceable
def rag_bot(question:str)->dict:
    #relevant content
    docs=retriever.invoke(question)
    doc_string=" ".join(doc.page_content for doc in docs)

    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions.       Use the following source documents to answer the user's questions.       If you don't know the answer, just say that you don't know.       Use three sentences maximum and keep the answer concise.

Documents:
{doc_string}"""
    
    ## llm invoke

    ai_msg=llm.invoke([
         {"role": "system", "content": instructions},
        {"role": "user", "content": question},

    ])
    return {"answer":ai_msg.content,"documents":docs}

In [16]:
def ls_target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])

In [17]:
print(ls_target({"question": "What is prompt engineering?"}))

{'answer': 'Prompt engineering, also known as In-Context Prompting, refers to methods for communicating with large language models (LLMs) to steer their behavior for desired outcomes without updating the model weights. It is an empirical science that requires heavy experimentation and heuristics to achieve the desired results. The goal of prompt engineering is about alignment and model steerability, aiming to control the output of LLMs through carefully crafted input prompts.', 'documents': [Document(id='2d1bb80f-f7c6-4821-b856-da42d45127b6', metadata={'source': 'https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/', 'title': "Prompt Engineering | Lil'Log", 'description': 'Prompt Engineering, also known as In-Context Prompting, refers to methods for how to communicate with LLM to steer its behavior for desired outcomes without updating the model weights. It is an empirical science and the effect of prompt engineering methods can vary a lot among models, thus requiring heav

In [18]:
rag_bot("What is agents")

{'answer': 'In the context of the provided documents, agents refer to autonomous entities controlled by a Large Language Model (LLM) that can perform tasks, make decisions, and interact with their environment. These agents are designed to simulate human-like behavior and can be used in various applications, such as virtual characters or problem-solving systems. They are essentially virtual entities that can think, learn, and act based on their programming and environment.',
 'documents': [Document(id='007db4c1-3dea-4d35-88fb-159e21642a1c', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be fram

Dataset

In [19]:
from langsmith import Client

client=Client()

# Define the examples for the dataset
examples = [
    {
        "inputs": {"question": "How does the ReAct agent use self-reflection? "},
        "outputs": {"answer": "ReAct integrates reasoning and acting, performing actions - such tools like Wikipedia search API - and then observing / reasoning about the tool outputs."},
    },
    {
        "inputs": {"question": "What are the types of biases that can arise with few-shot prompting?"},
        "outputs": {"answer": "The biases that can arise with few-shot prompting include (1) Majority label bias, (2) Recency bias, and (3) Common token bias."},
    },
    {
        "inputs": {"question": "What are five types of adversarial attacks?"},
        "outputs": {"answer": "Five types of adversarial attacks are (1) Token manipulation, (2) Gradient based attack, (3) Jailbreak prompting, (4) Human red-teaming, (5) Model red-teaming."},
    }
]

### create the daatset and example in LAngsmith
dataset_name="RAGG Test Evaluation"
dataset = client.create_dataset(dataset_name=dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=examples
)

{'example_ids': ['de7d721f-86b3-4350-a9b1-a7621f2dd867',
  '0d4b7de5-5325-4626-8637-2ea845b47d7f',
  'd9f416be-c820-4f94-8ec1-b5d77f3e736a'],
 'count': 3,
 'as_of': '2026-06-12T07:19:17.267914426Z'}

Evaluators or Metrics

Correctness: Response vs reference answer

Goal: Measure "how similar/correct is the RAG chain answer, relative to a ground-truth answer"
Mode: Requires a ground truth (reference) answer supplied through a dataset

Evaluator: Use LLM-as-judge to assess answer correctness.

In [20]:
from typing_extensions import Annotated,TypedDict

## Correctness Output Schema

# Grade output schema
class CorrectnessGrade(TypedDict):
    # Note that the order in the fields are defined is the order in which the model will generate them.
    # It is useful to put explanations before responses because it forces the model to think through
    # its final response before generating it:
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]

## correctness prompt

correctness_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. 
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the  ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

from langchain_groq import ChatGroq

grader_llm=ChatGroq(model="llama-3.3-70b-versatile",temperature=0).with_structured_output(CorrectnessGrade,
                                                                         method="json_schema",strict=True)
## evaluator
def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """An evaluator for RAG answer accuracy"""
    answers = f"""\
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['answer']}"""

    # Run evaluator
    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions}, 
        {"role": "user", "content": answers}
    ])
    return grade["correct"]

Relevance: Response vs input

The flow is similar to above, but we simply look at the inputs and outputs without needing the reference_outputs. Without a reference answer we can't grade accuracy, but can still grade relevance—as in, did the model address the user's question or not.

In [21]:
# Grade output schema
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "Provide the score on whether the answer addresses the question"]

# Grade prompt
relevance_instructions="""You are a teacher grading a quiz. 

You will be given a QUESTION and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM
relevance_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0).with_structured_output(RelevanceGrade, method="json_schema", strict=True)

# Evaluator
def relevance(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness."""
    answer = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]


Groundedness: Response vs retrieved docs

Another useful way to evaluate responses without needing reference answers is to check if the response is justified by (or "grounded in") the retrieved documents.

In [22]:
# Grade output schema
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[bool, ..., "Provide the score on if the answer hallucinates from the documents"]

# Grade prompt
grounded_instructions = """You are a teacher grading a quiz. 

You will be given FACTS and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS. 
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM 
grounded_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0).with_structured_output(GroundedGrade, method="json_schema", strict=True)

# Evaluator
def groundedness(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer groundedness."""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = grounded_llm.invoke([{"role": "system", "content": grounded_instructions}, {"role": "user", "content": answer}])
    return grade["grounded"]

Retrieval Relevance: Retrieved docs vs input

In [23]:
# Grade output schema
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the retrieved documents are relevant to the question, False otherwise"]

# Grade prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a set of FACTS provided by the student. 

Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM
retrieval_relevance_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0).with_structured_output(RetrievalRelevanceGrade, method="json_schema", strict=True)

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance"""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"

    # Run evaluator
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]


Run the evaluation

In [24]:
def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])

experiment_results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness, groundedness, relevance, retrieval_relevance],
    experiment_prefix="rag-doc-relevance",
    metadata={"version": "LCEL context, gpt-4-0125-preview"},
)
# Explore results locally as a dataframe if you have pandas installed
experiment_results.to_pandas()

View the evaluation results for experiment: 'rag-doc-relevance-e4efea22' at:
https://smith.langchain.com/o/87d68caf-a391-48ca-a2c6-1fc4f60b9445/datasets/06797f9a-2586-406b-8e84-bb453159edf6/compare?selectedSessions=d5fa56c5-a95a-4c08-87af-7cbe9c543acf




0it [00:00, ?it/s]Error running evaluator <DynamicRunEvaluator correctness> on run 019ebab3-1406-79d2-9b9d-2392a3e006a7: BadRequestError("Error code: 400 - {'error': {'message': 'This model does not support response format `json_schema`. See supported models at https://console.groq.com/docs/structured-outputs#supported-models', 'type': 'invalid_request_error', 'param': 'response_format'}}")
Traceback (most recent call last):
  File "c:\Users\Omkar Kadam\Desktop\Agentic AI\.venv\Lib\site-packages\langsmith\evaluation\_runner.py", line 1637, in _run_evaluators
    evaluator_response = evaluator.evaluate_run(  # type: ignore[call-arg]
        run=run,
        example=example,
        evaluator_run_id=evaluator_run_id,
    )
  File "c:\Users\Omkar Kadam\Desktop\Agentic AI\.venv\Lib\site-packages\langsmith\evaluation\evaluator.py", line 332, in evaluate_run
    result = self.func(
        run,
        example,
        langsmith_extra={"run_id": evaluator_run_id, "metadata": metadata},
    )

,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.wrapper,execution_time,example_id,id
0,What are the types of biases that can arise wi...,"According to Zhao et al. (2021), three biases ...",[page_content='Text: i'll bet the video game i...,None,The biases that can arise with few-shot prompt...,None,0.556576,0d4b7de5-5325-4626-8637-2ea845b47d7f,019ebab3-1406-79d2-9b9d-2392a3e006a7
1,What are five types of adversarial attacks?,The five types of adversarial attacks are: \n1...,[page_content='Adversarial attacks are inputs ...,None,Five types of adversarial attacks are (1) Toke...,None,0.364607,d9f416be-c820-4f94-8ec1-b5d77f3e736a,019ebab3-18cd-7a22-b929-cb1cc383cdb8
2,How does the ReAct agent use self-reflection?,The ReAct agent uses self-reflection by integr...,[page_content='Self-reflection is a vital aspe...,None,"ReAct integrates reasoning and acting, perform...",None,0.630932,de7d721f-86b3-4350-a9b1-a7621f2dd867,019ebab3-1bdf-7b21-aa64-082b857ace7f
